# TP 6 — DataFrame, Catalyst et lecture de plans### Module 4 — Spark SQL et optimisation**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. **Mesurer** l'écart entre RDD et DataFrame en PySpark, et l'expliquer.2. Mesurer les trois façons d'appliquer une fonction : native, `pandas_udf`, UDF simple.3. Lire un plan `EXPLAIN FORMATTED` et y repérer projection, pushdown et shuffles.4. **Démontrer** que Catalyst réordonne — et trouver le cas où il ne le fait pas.5. Vérifier que SQL et API DataFrame produisent le même plan.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Préparation | 1 || 2 | RDD contre DataFrame | 4 || 3 | Les trois façons d'appliquer une fonction | 5 || 4 | Lire un plan | 5 || 5 | Catalyst réordonne — et son mur | 5 |

---# Exercice 1 — Préparation  *(1 point)*

In [ ]:
from pyspark.sql import SparkSession, functions as Ffrom pyspark.sql.types import DoubleTypeimport timeFILIERE = "if"          # "if" ou "an"UTILISATEUR = "etudiant"spark = (SparkSession.builder         .appName("TP6 - Catalyst")         .master("local[4]")         .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")         .config("spark.sql.shuffle.partitions", "16")         .getOrCreate())spark.sparkContext.setLogLevel("WARN")print("Spark", spark.version, "| UI :", spark.sparkContext.uiWebUrl)def chrono(libelle, fonction):    debut = time.time()    resultat = fonction()    duree = time.time() - debut    print(f"{libelle:<38} {duree:7.2f} s")    return duree, resultat

In [ ]:
# 1.2 — Les données, en Parquet trié (produites au TP2)COL_CLE = "pays_transaction" if FILIERE == "if" else "pays"COL_NUM = "montant"          if FILIERE == "if" else "position_s"COL_TRI = "horodatage"CHEMIN = f"hdfs://namenode:8020/user/{UTILISATEUR}/formats/pq_trie"df = spark.read.parquet(CHEMIN)n = df.count()print(f"{n:,} lignes")df.printSchema()

> **Si le répertoire n'existe pas**, relancez la cellule 5.1 du TP2, ou écrivez-le ici :> `spark.read.json(source).orderBy("horodatage").write.parquet(CHEMIN)`

---# Exercice 2 — RDD contre DataFrame  *(4 points)***Objectif.** Mesurer l'écart, et comprendre qu'il ne tient pas seulement à Catalyst.

## 2.1 — **PRÉDICTION** *(1 pt)*Les deux cellules suivantes calculent la **même somme par clé**. Laquelle sera la plus rapide,et de quel ordre de grandeur ? Citez **deux** raisons distinctes.

**Votre réponse :***(rédigez ici)*

In [ ]:
# 2.2 — Version RDDdef somme_rdd():    return (df.rdd              .map(lambda r: (r[COL_CLE], r[COL_NUM] or 0.0))              .reduceByKey(lambda a, b: a + b)              .collect())t_rdd, r1 = chrono("RDD  : map + reduceByKey", somme_rdd)

In [ ]:
# 2.3 — Version DataFramedef somme_df():    return df.groupBy(COL_CLE).agg(F.sum(COL_NUM)).collect()t_df, r2 = chrono("DataFrame : groupBy + sum", somme_df)print(f"\nrapport : {t_rdd/t_df:.1f}x")

In [ ]:
# 2.4 — Ce que chacun a réellement luprint("=== plan du DataFrame ===")df.groupBy(COL_CLE).agg(F.sum(COL_NUM)).explain(mode="formatted")

### Q2 *(3 pts)* —- **a.** Quel rapport avez-vous mesuré ?- **b.** Dans le plan du DataFrame, regardez `ReadSchema`. Combien de colonnes sont lues ?  Et dans la version RDD, combien le sont nécessairement ?- **c.** Dans la Spark UI, comparez le *Shuffle Write* des deux jobs. Que constatez-vous, et  quel mécanisme du module 3 reconnaissez-vous dans le plan du DataFrame ?

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Trois façons d'appliquer une fonction  *(5 points)***Objectif.** Mesurer le coût d'une UDF Python, et celui de ses deux alternatives.On calcule un indicateur simple : une catégorie de montant.

In [ ]:
# 3.1 — Version 1 : UDF Python, ligne à lignefrom pyspark.sql.functions import udffrom pyspark.sql.types import StringType@udf(returnType=StringType())def categorie_udf(valeur):    if valeur is None:        return "inconnu"    if valeur < 20:    return "micro"    if valeur < 100:   return "petit"    if valeur < 1000:  return "moyen"    return "grand"t_udf, _ = chrono("1. UDF Python ligne a ligne",                  lambda: df.withColumn("cat", categorie_udf(F.col(COL_NUM)))                            .groupBy("cat").count().collect())

In [ ]:
# 3.2 — Version 2 : pandas_udf (vectorisée, via Arrow)from pyspark.sql.functions import pandas_udfimport pandas as pd@pandas_udf(StringType())def categorie_pandas(s: pd.Series) -> pd.Series:    return pd.cut(s.fillna(-1),                  bins=[-2, -0.5, 20, 100, 1000, float("inf")],                  labels=["inconnu", "micro", "petit", "moyen", "grand"]).astype(str)t_pandas, _ = chrono("2. pandas_udf (vectorisee)",                     lambda: df.withColumn("cat", categorie_pandas(F.col(COL_NUM)))                               .groupBy("cat").count().collect())

In [ ]:
# 3.3 — Version 3 : expressions nativescategorie_native = (F.when(F.col(COL_NUM).isNull(), "inconnu")                     .when(F.col(COL_NUM) < 20,   "micro")                     .when(F.col(COL_NUM) < 100,  "petit")                     .when(F.col(COL_NUM) < 1000, "moyen")                     .otherwise("grand"))t_natif, _ = chrono("3. Expressions natives",                    lambda: df.withColumn("cat", categorie_native)                              .groupBy("cat").count().collect())print(f"\nUDF / natif        : {t_udf/t_natif:.1f}x")print(f"UDF / pandas_udf   : {t_udf/t_pandas:.1f}x")print(f"pandas_udf / natif : {t_pandas/t_natif:.1f}x")

In [ ]:
# 3.4 — Comparer les plansprint("=== UDF Python ===")df.withColumn("cat", categorie_udf(F.col(COL_NUM))).explain(mode="formatted")

In [ ]:
print("=== natif ===")df.withColumn("cat", categorie_native).explain(mode="formatted")

### Q3 *(5 pts)* —- **a.** Reportez vos trois mesures et les trois rapports.- **b.** Dans le plan de l'UDF Python, quel opérateur apparaît que l'on ne trouve pas dans le  plan natif ? Que signifie-t-il ?- **c.** Dans le plan natif, le `when ... otherwise` apparaît-il comme un opérateur séparé, ou  est-il intégré ? Que cela signifie-t-il pour l'optimisation ?- **d.** L'écart mesuré ici est-il représentatif de ce qu'on observerait en production ?  Justifiez en pensant à ce que `local[4]` a de particulier.- **e.** Formulez la règle pratique à retenir, en une phrase.

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Lire un plan  *(5 points)***Objectif.** Savoir extraire d'un plan les trois informations qui comptent.

In [ ]:
# 4.1 — Une requête complèterequete = (df           .filter(F.col(COL_NUM) > 500)           .filter(F.col(COL_TRI) >= "2025-02-01")           .groupBy(COL_CLE)           .agg(F.sum(COL_NUM).alias("total"),                F.count("*").alias("nb"))           .orderBy(F.desc("total")))requete.explain(mode="formatted")

In [ ]:
# 4.2 — Exécuter, puis relever les métriquest, res = chrono("requete complete", lambda: requete.collect())for r in res[:5]:    print(r)print("\nRelevez dans la Spark UI, onglet SQL :")print("  - number of files read")print("  - size of files read")print("  - number of output rows du FileScan")

### Q4 *(5 pts)* — À partir du plan et des métriques :- **a.** Combien d'`Exchange` ce plan contient-il ? Combien de stages cela implique-t-il ?- **b.** Quelles colonnes le `ReadSchema` contient-il ? La table en compte huit. Que s'est-il  passé ?- **c.** Quels filtres apparaissent dans `PushedFilters` ? Les deux y sont-ils ?- **d.** Combien de fichiers ont été lus, sur combien au total ? Le pushdown a-t-il été  **efficace** ? Rattachez à ce que vous avez mesuré au TP2.- **e.** Repérez le double `HashAggregate`. Que fait le premier, et quel concept du module 3  cela illustre-t-il ?

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Catalyst réordonne, et son mur  *(5 points)***Objectif.** Démontrer que l'ordre d'écriture importe peu — puis trouver l'exception.

In [ ]:
# 5.1 — Deux écritures, ordre inverséordre_a = (df.filter(F.col(COL_TRI) >= "2025-02-01")             .withColumn("double", F.col(COL_NUM) * 2)             .select(COL_CLE, "double"))ordre_b = (df.withColumn("double", F.col(COL_NUM) * 2)             .select(COL_CLE, "double", COL_TRI)             .filter(F.col(COL_TRI) >= "2025-02-01")             .select(COL_CLE, "double"))print("=== ORDRE A : filtre d'abord ===")ordre_a.explain(mode="formatted")

In [ ]:
print("=== ORDRE B : filtre en dernier ===")ordre_b.explain(mode="formatted")

In [ ]:
# 5.2 — Mesurer les deuxta, _ = chrono("ordre A (filtre d'abord)",  lambda: ordre_a.count())tb, _ = chrono("ordre B (filtre en dernier)", lambda: ordre_b.count())print(f"\nrapport : {tb/ta:.2f}x")

### Q5a *(2 pts)* — Les deux plans physiques sont-ils identiques ? Les temps le sont-ils ?Que démontrez-vous ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 5.3 — Le mur : la même chose, avec une UDF au milieu@udf(returnType=DoubleType())def double_udf(v):    return (v or 0.0) * 2avec_udf = (df.withColumn("double", double_udf(F.col(COL_NUM)))              .filter(F.col(COL_TRI) >= "2025-02-01")              .select(COL_CLE, "double"))print("=== UDF AVANT le filtre ===")avec_udf.explain(mode="formatted")

In [ ]:
# 5.4 — Et en déplaçant simplement le filtre avant l'UDFudf_apres = (df.filter(F.col(COL_TRI) >= "2025-02-01")               .withColumn("double", double_udf(F.col(COL_NUM)))               .select(COL_CLE, "double"))print("=== filtre AVANT l'UDF ===")udf_apres.explain(mode="formatted")t1, _ = chrono("UDF avant le filtre", lambda: avec_udf.count())t2, _ = chrono("filtre avant l'UDF",  lambda: udf_apres.count())print(f"\nrapport : {t1/t2:.1f}x")

### Q5b *(3 pts)* —- **a.** Dans le plan 5.3, où se trouve le `Filter` par rapport au `BatchEvalPython` ?  Catalyst a-t-il pu le remonter ?- **b.** Comparez les temps de 5.4. Quel écart ?- **c.** Ce résultat contredit-il ce que vous avez conclu en Q5a ? Formulez la règle complète,  qui couvre les deux cas.

**Votre réponse :***(rédigez ici)*

---# Synthèse| Mesure | Votre chiffre | Explication en une ligne ||---|---|---|| RDD / DataFrame | | || UDF / natif | | || UDF / `pandas_udf` | | || Fichiers lus / total (Q4d) | | || UDF avant / après le filtre | | |**Question de conclusion.** Parmi tout ce que vous avez mesuré, quelle correction offre lemeilleur rapport entre effort et gain ?

In [ ]:
spark.stop()print("Session fermée.")

---## Avant de rendre- [ ] Les **deux prédictions** (2.1 et votre estimation en Q3) sont écrites avant exécution.- [ ] Les questions **Q2 à Q5** sont rédigées, avec vos chiffres.- [ ] Le tableau de synthèse est complété.- [ ] Notebook exporté en HTML et déposé.**Bon TP.**